# Review: long-read proteogenomics of one comparison

This page gathers the results of one comparison between two groups of samples, from the long-read (PacBio Iso-Seq) RNA sequencing run through the LRP2 pipeline. It is for review: it shows the figures and tables that the other notebooks in this folder made, and counts the rows of the pipeline's own statistical tables. It computes no new statistics. Under every figure is a short note on how to read it. The last sections say what this analysis does not show, and how every step was done.

The unit of the analysis is the **measured full-length isoform** and the **ORF** (predicted protein form) it carries, not the gene. Long reads show whole transcripts, so isoforms that are not in the reference annotation (**novel**) are measured the same way as known ones.

In [ ]:
source("src/paths.R")
source("src/heatmap.R")
source("src/review.R")
options(repr.plot.width = 8, repr.plot.height = 5)
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison
alpha <- 0.05                 # false discovery rate (FDR) below which a difference is called significant
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))
lev <- sort(unique(groups$group))
control_group <- lev[1]; case_group <- lev[2]    # edgeR's order: logFC above 0 means higher in the second group
# Things the reader must know before reading the figures. Edit this list for the comparison being shown.
notes <- c(
  "All NoSpA samples are B27-negative, so this comparison cannot separate the effect of SpA from the effect of B27 status.",
  "The groups are small and unequal, so a result can rest on one or two samples; the figures show every sample."
)
cat("Comparison:", comparison, "\n"); cat(paste0("- ", notes), sep = "\n")
table(groups$group)

**Explanation.** `paths.json` says which run this page is about, as in the other notebooks. `alpha` is the false discovery rate used throughout: the share of results called significant that we expect to be false. edgeR's second group, here `case_group`, is the one for which a log fold change (logFC) above 0 means higher expression. `notes` is a list of facts the reader needs before the figures; it is written by hand for each comparison. The table gives the number of samples in each group.

## 1. How many genes, isoforms and ORFs differ between the groups?

In [ ]:
de_summary(loc, comparison, alpha = alpha, lfc = 1, up_in = case_group, down_in = control_group)

**Explanation.** `tested` is how many rows edgeR tested at each level. `significant` is how many have an FDR below `alpha`; the next two columns split them by the group in which they are higher. The last column keeps only those that are also at least two-fold different (log2 fold change of 1). The numbers are read from the pipeline's own edgeR tables (`*_DGE_`, `*_DTE_` and `*_DE_ORF_edgeR_results.txt`). Genes are listed for comparison only; the isoform and ORF rows are the unit of this analysis.

## 2. Volcano plots

In [ ]:
show_fig("volcano_genes", "volcano.ipynb")

In [ ]:
show_fig("volcano_isoforms", "volcano.ipynb")

In [ ]:
show_fig("volcano_orfs", "volcano.ipynb")

**How to read.** Each point is a gene, an isoform or an ORF. Left to right is how much higher it is in one group than the other (log2 fold change); height is how sure we are (minus log10 of the FDR). Dashed lines mark FDR 0.05 and a two-fold change. For isoforms and ORFs the points that pass both are coloured **known** (the same splice junctions as a known transcript), **novel** (anything else) or **no ORF** (no predicted protein), and the legend counts them; grey points do not pass. Which group is on the right is stated under the heading of section 1.

## 3. Heatmaps of the isoforms and ORFs that differ

In [ ]:
show_fig("isoform_heatmap", "heatmaps.ipynb")

In [ ]:
show_fig("orf_heatmap", "heatmaps.ipynb")

**How to read.** One row per isoform or ORF that edgeR found different (FDR below `alpha`), one column per sample, grouped by group. Colour is the row's expression relative to its own average (orange higher, blue lower). A real group difference is the same colour across one whole group and the other colour across the other. The bar at the right shows known or novel. A row that is orange in only one or two samples is not a group difference, whatever its p value.

## 4. Isoforms and ORFs found in one group and not the other

In [ ]:
show_fig("specific_counts", "specific_variants.ipynb")

In [ ]:
show_fig("specific_novelty", "specific_variants.ipynb")

**How to read.** A feature is called *specific* to a group when it is detected in at least two samples of that group and in none of the other. This is a presence and absence count; **it is not a statistical test**. A larger group has more chances to detect a rare feature, so when the groups differ in size more features come out specific to the larger group by chance alone. A feature that is absent in the other group may only be unsampled, because long-read depth per isoform is shallow. Treat these as leads to check, not as findings.

## 5. Splicing: clusters of splice junctions

In [ ]:
show_fig("splicing_volcano", "splicing.ipynb")

In [ ]:
show_fig("splicing_heatmap", "splicing.ipynb")

**How to read.** This looks at splice junctions directly, without naming isoforms (leafcutter, run on the long reads). Each point of the volcano is a cluster of junctions; the horizontal axis is the change in the share of reads that use a junction (delta PSI, percent spliced in), the height is the FDR. The heatmap shows the junctions of the clusters that differ at FDR below `alpha`.

## 6. Usage: does a gene use its isoforms differently?

In [ ]:
show_fig("usage_isoforms_pvalues", "usage.ipynb")

In [ ]:
show_fig("usage_isoforms_panel", "usage.ipynb")

In [ ]:
show_fig("usage_orfs_panel", "usage.ipynb")

**How to read.** Differential *usage* asks whether the share of a gene's reads that go to each isoform (or ORF) differs between the groups, even when the gene's total does not (DRIMSeq). The histogram is the raw p value of every tested gene; a pile-up near zero means real signal, a flat histogram means little. The panels draw only the genes with an adjusted p value below `alpha`; when there are none, the notebook says so and nothing is drawn.

## 7. The dominant ORF of each gene

In [ ]:
dom <- read_saved_table("dominant_orf", "dominant_orf.ipynb")
if (!is.null(dom)) head(dom, 15)

In [ ]:
show_fig("dominant_orf_heatmap", "dominant_orf.ipynb")

**How to read.** Many isoforms of a gene can code for the same protein form, so the notebook counts per ORF. The *dominant ORF* of a gene is the one with the highest mean normalized expression, and its share is its part of the gene's ORF expression. A *switch* is a gene whose dominant ORF is a different one in the two groups. The table shows the first 15 rows; the whole table is in `tables/`.

## 8. Isoforms and ORFs that move together (WGCNA)

In [ ]:
show_fig("wgcna_module_group", "wgcna_isoforms.ipynb")

In [ ]:
show_fig("wgcna_eigengenes", "wgcna_isoforms.ipynb")

In [ ]:
show_fig("wgcna_orf_module_group", "wgcna_orfs.ipynb")

In [ ]:
show_fig("wgcna_orf_eigengenes", "wgcna_orfs.ipynb")

**How to read.** WGCNA groups isoforms (or ORFs) whose expression rises and falls together across the samples into *modules*, then tests each module's summary (its eigengene) against the group. A bar to the right is higher in `case_group`; a star marks a module with FDR below `alpha` after correcting for the number of modules. This is exploratory: with this many samples the modules are not stable, and a module is a statement about co-expression, not about cause. The heatmap shows the module summaries sample by sample; check that a starred module is not driven by one sample.

## What this analysis does not show

- **The groups differ in more than one way** when stated under `notes` above; the comparison cannot tell which difference a result follows.
- **Small groups.** With a handful of samples per group a result can rest on one or two samples. Significant means unlikely by chance under the model, not large or important.
- **Absent is not proven absent.** Long-read depth per isoform is shallow; a feature seen in no sample of one group may be unsampled there.
- **Specific is not significant.** Section 4 counts presence and absence only; the tests are in sections 1, 5 and 6.
- **Known and novel** are defined by the SQANTI class (`full-splice_match` is known; everything else, including incomplete copies of known transcripts, is novel). A novel isoform is a candidate, not a validated isoform.
- **No protein-level measurement.** ORFs are predicted from the transcript sequence; no mass spectrometry is used here.
- **WGCNA is exploratory** (section 8); it has not been tested for stability or preservation.

## How each step was done

Every method below is the one the pipeline or notebook ran; settings are those of the run (see the S4 logs and the notebooks).

- **Differential expression of genes, isoforms and ORFs:** edgeR exact test on counts, genes and isoforms or ORFs with too few reads removed by `filterByExpr`, TMM normalization, robust dispersion estimate, Benjamini-Hochberg FDR. Robinson MD, McCarthy DJ, Smyth GK. *Bioinformatics* 2010;26:139-140, doi:10.1093/bioinformatics/btp616. Robinson MD, Oshlack A. *Genome Biology* 2010;11:R25, doi:10.1186/gb-2010-11-3-r25.
- **Differential usage of isoforms and ORFs:** DRIMSeq, a Dirichlet-multinomial model, after its filter for low expression. Nowicka M, Robinson MD. *F1000Research* 2016;5:1356, doi:10.12688/f1000research.8900.2.
- **Splice-junction clusters:** leafcutter, run on the long reads. Li YI et al. *Nature Genetics* 2018;50:151-158, doi:10.1038/s41588-017-0004-9.
- **Known and novel:** SQANTI classification of the isoforms and of the predicted proteins. Tardaguila M et al. *Genome Research* 2018;28:396-411, doi:10.1101/gr.222976.117.
- **Specific features:** detected in at least two samples of one group and none of the other; counts only (this page, section 4).
- **WGCNA:** read-count filter, PFlog1pPF normalization, signed network, modules and eigengene correlation, as in the endotypes-transcriptomics notebooks; Langfelder P, Horvath S. *BMC Bioinformatics* 2008;9:559, doi:10.1186/1471-2105-9-559; Booeshaghi AS et al. *bioRxiv* 2022, doi:10.1101/2022.05.06.490859.
- **Multiple testing:** Benjamini-Hochberg false discovery rate throughout, at the `alpha` set at the top.

In [ ]:
si <- Sys.info()
cat("run on :", si[["nodename"]], "(", si[["sysname"]], si[["release"]], ")\n")
cat("date   :", format(Sys.time(), "%Y-%m-%d %H:%M %Z"), "\n")
cat("R      :", R.version.string, "\n")
cat("run    :", loc$run_dir, "\n")
for (p in c("data.table", "ComplexHeatmap", "jsonlite")) cat(sprintf("  %-16s %s\n", p, as.character(packageVersion(p))))

**Explanation.** Records where and with what versions this ran, and which run it describes. It is the same in every notebook.